In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
from pyspark.sql import functions as F

players = spark.table("lh_silver_game.players_clean")
sessions = spark.table("lh_silver_game.sessions_clean")
purchases = spark.table("lh_silver_game.purchases_clean")
ad_events = spark.table("lh_silver_game.ad_events_clean")

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 3, Finished, Available, Finished, False)

In [2]:
sessions_daily = (
    sessions
    .withColumn(
        "activity_date",
        F.to_date("session_start")
    )
)

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 4, Finished, Available, Finished, False)

In [3]:
daily_session_metrics = (
    sessions_daily
    .groupBy("activity_date")
    .agg(
        F.countDistinct("player_id").alias("dau"),
        F.count("*").alias("total_sessions"),
        F.sum("session_duration_minutes").alias("total_session_minutes")
    )
    .withColumn(
        "avg_sessions_per_active_user",
        F.col("total_sessions") / F.col("dau")
    )
)

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 5, Finished, Available, Finished, False)

In [4]:
display(
    daily_session_metrics
    .orderBy("activity_date")
    .limit(10)
)

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9c282069-bd4f-4157-9b84-d41005f0f9d7)

In [5]:
purchases_daily = (
    purchases
    .withColumn(
        "activity_date",
        F.to_date("timestamp")
    )
)

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 7, Finished, Available, Finished, False)

In [6]:
daily_purchase_metrics = (
    purchases_daily
    .groupBy("activity_date")
    .agg(
        F.sum("price_usd").alias("purchase_revenue"),
        F.countDistinct("player_id").alias("payer_count"),
        F.count("*").alias("purchase_count")
    )
)

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 8, Finished, Available, Finished, False)

In [7]:
display(
    daily_purchase_metrics
    .orderBy("activity_date")
    .limit(10)
)

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e7b963df-6ae7-4a3b-804c-cf57c2167ef9)

In [8]:
ad_daily = (
    ad_events
    .withColumn(
        "activity_date",
        F.to_date("timestamp")
    )
)

daily_ad_metrics = (
    ad_daily
    .groupBy("activity_date")
    .agg(
        F.sum("revenue_usd").alias("ad_revenue"),
        F.count("*").alias("ad_event_count")
    )
)

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 10, Finished, Available, Finished, False)

In [9]:
display(
    daily_ad_metrics
    .orderBy("activity_date")
    .limit(10)
)

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 78b4d670-df61-4649-800c-d8588fea4197)

In [10]:
daily_installs = (
    players
    .groupBy(
        F.col("install_date").alias("activity_date")
    )
    .agg(
        F.count("*").alias("new_installs")
    )
)

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 12, Finished, Available, Finished, False)

In [11]:
display(
    daily_installs
    .orderBy("activity_date")
    .limit(10)
)

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9a5a9489-79b1-4acc-9654-7098a7221ff1)

In [12]:
daily_kpis = (
    daily_session_metrics
    .join(daily_purchase_metrics, on="activity_date", how="left")
    .join(daily_ad_metrics, on="activity_date", how="left")
    .join(daily_installs, on="activity_date", how="left")
)

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 14, Finished, Available, Finished, False)

In [13]:
daily_kpis = daily_kpis.fillna({
    "purchase_revenue": 0.0,
    "payer_count": 0,
    "purchase_count": 0,
    "ad_revenue": 0.0,
    "ad_event_count": 0,
    "new_installs": 0
})

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 15, Finished, Available, Finished, False)

In [14]:
daily_kpis = (
    daily_kpis
    .withColumn(
        "total_revenue",
        F.col("purchase_revenue") + F.col("ad_revenue")
    )
)

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 16, Finished, Available, Finished, False)

In [15]:
display(
    daily_kpis
    .orderBy("activity_date")
    .limit(10)
)

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, daa256e6-7202-4b1c-8090-7861c818387a)

In [17]:
daily_kpis.write.format("delta").mode("overwrite").saveAsTable("lh_gold_game.daily_kpis")

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 19, Finished, Available, Finished, False)

In [18]:
df_check = spark.table("lh_gold_game.daily_kpis")

print("Saved row count:", df_check.count())
display(df_check.orderBy("activity_date").limit(10))

StatementMeta(, d82fcddc-20d5-48ec-90ea-09ad383c4280, 20, Finished, Available, Finished, False)

Saved row count: 90


SynapseWidget(Synapse.DataFrame, 19a97ee7-bde0-482a-85b9-b5524fb7ed15)